# Computer Exercise 15.26 — Problem 2

> **교재**: Cheney & Kincaid, *Numerical Mathematics and Computing* (7th ed.) — 확장 사례연구
> **단원**: §15.26 Sequential Decision Making — *Cramér vs KL under Real-Net Bellman Projection*
> **풀이 일자**: Day 93
> **언어**: Python 3 (NumPy / Matplotlib)


## 1. 문제 (원문)

> **Problem 2.** Day 92 (§15.25 Problem 2) reported that in a *stationary* Gaussian-mixture
> categorical projection sandbox, the **KL** loss beat both the Cramér closed-form and the
> 1-Wasserstein sign-subgradient surrogates on sharpness and mean-tracking bias. That verdict was
> obtained without any Bellman bootstrapping — the target was fixed. Re-adjudicate the
> Cramér vs KL comparison inside a **real Bellman-projection loop**: at each step,
> $Z^\pi(s, a) \stackrel{D}{=} R + \gamma Z^\pi(s', \pi(s'))$, and the categorical head must
> match the *projected* distribution of the bootstrapped return. Report training-time atom
> entropy $H(\hat p_t)$, projection error (KL against the projected target $m$), and evaluation
> return, under matched budget and seeds. Additionally ablate the **atom count**
> $K \in \{10, 21, 51\}$ to check whether the KL advantage is $K$-independent.

### 한국어 풀이용 정리
Day 92 P2 는 target 이 정지 (stationary). §15.26 P2 는 그 손실 비교를 Bellman projection loop
안에서 재판정. 5-상태 chain MDP 에서 TD-target 을 categorical projection
$\Phi(r + \gamma Z(s', \pi(s')))$ 로 만든 뒤 Cramér / KL 각각 학습, sharpness · projection
error · evaluation return 비교. K 를 3 수준으로 sweep.


## 2. 수학적 배경

### 2.1 분포적 Bellman operator
$\mathcal T^\pi Z(s,a) \stackrel{D}{=} R + \gamma Z(s', a')$, $a' = \arg\max_{a'} \mathbb E Z(s',a')$.

### 2.2 Categorical projection $\Phi$
$T_i = \text{clip}(r + \gamma z_i, v_{\min}, v_{\max})$,
$b = (T_i - v_{\min}) / \Delta z$, $l = \lfloor b \rfloor$, $u = \lceil b \rceil$,
$m_l \mathrel{+}= p_i (u - b)$, $m_u \mathrel{+}= p_i (b - l)$.

### 2.3 두 손실
- **Cramér**: $L = \sum_k (P_k - M_k)^2$, $P, M$ = CDF.
- **KL**: $L = -\sum_k m_k \log \hat p_k$.

### 2.4 지표
- **Sharpness**: $H(\hat p) = -\sum_k \hat p_k \log \hat p_k$.
- **Projection fit**: $-\sum_k m_k \log \hat p_k$ (cross-entropy).
- **Evaluation return**: greedy 배포 tail-8.


## 3. 풀이 흐름

1. 확률적 chain MDP + tabular per-(s,a) $K$-원자 logit head.
2. TD-target: greedy next action, categorical projection.
3. 두 손실 각각으로 800 step 학습, 3 시드.
4. K sweep $\{10, 21, 51\}$.
5. sharpness / projection error / eval return 표 + 그래프.
6. Reference: MC empirical return distribution 과 학습된 원자 분포 비교.


In [1]:
import os
os.environ['MPLCONFIGDIR'] = '/tmp/mplcfg'
os.makedirs('/tmp/mplcfg', exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

pd.set_option("display.float_format", lambda v: f"{v:.4f}")

class ChainMDP:
    def __init__(self, N=5, p_slip=0.10, step_r=-0.02, goal_r=1.0, rng=None):
        self.N, self.p_slip, self.step_r, self.goal_r = N, p_slip, step_r, goal_r
        self.rng = rng or np.random.default_rng(0)
    def reset(self):
        self.s = 0; return self.s
    def step(self, a):
        if self.rng.random() < self.p_slip: a = 1 - a
        if a == 1: self.s = min(self.s + 1, self.N - 1)
        else:      self.s = max(self.s - 1, 0)
        done = (self.s == self.N - 1)
        r = self.goal_r if done else self.step_r
        return self.s, r, done

def softmax(z, axis=-1):
    z = z - np.max(z, axis=axis, keepdims=True)
    e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)

def entropy(p, eps=1e-12):
    return float(-np.sum(p * np.log(p + eps)))

print("env ready")


env ready


In [2]:
def categorical_projection(r, p, atoms, gamma, done):
    K = len(atoms)
    v_min, v_max = atoms[0], atoms[-1]
    dz = (v_max - v_min) / (K - 1)
    if done:
        T = np.full(K, r)
    else:
        T = np.clip(r + gamma * atoms, v_min, v_max)
    m = np.zeros(K)
    b = (T - v_min) / dz
    lo = np.floor(b).astype(int)
    hi = np.ceil(b).astype(int)
    for i in range(K):
        if lo[i] == hi[i]:
            m[lo[i]] += p[i]
        else:
            m[lo[i]] += p[i] * (hi[i] - b[i])
            m[hi[i]] += p[i] * (b[i] - lo[i])
    m = m / (m.sum() + 1e-12)
    return m

def cramer_grad_logits(p, m):
    P = np.cumsum(p); M = np.cumsum(m)
    dL_dp = np.array([2 * np.sum(P[j:] - M[j:]) for j in range(len(p))])
    return p * (dL_dp - np.sum(dL_dp * p))

def kl_grad_logits(p, m):
    return p - m

print("projection & grads ready")


projection & grads ready


In [3]:
class CatLearner:
    def __init__(self, seed, N=5, A=2, K=21, v_min=-0.5, v_max=1.0,
                 loss="cramer", lr=0.2, gamma=0.95, eps=0.10):
        self.rng = np.random.default_rng(seed)
        self.N, self.A, self.K = N, A, K
        self.loss = loss; self.lr = lr; self.gamma = gamma; self.eps = eps
        self.atoms = np.linspace(v_min, v_max, K)
        self.logits = self.rng.normal(0, 0.1, size=(N, A, K))
    def p_dist(self, s, a):
        return softmax(self.logits[s, a])
    def q_value(self, s, a):
        return float(self.p_dist(s, a) @ self.atoms)
    def act(self, s):
        if self.rng.random() < self.eps:
            return int(self.rng.integers(0, self.A))
        return int(np.argmax([self.q_value(s, a) for a in range(self.A)]))
    def update(self, s, a, r, sp, done):
        if done:
            m = categorical_projection(r, np.zeros(self.K), self.atoms, self.gamma, True)
        else:
            ap = int(np.argmax([self.q_value(sp, a2) for a2 in range(self.A)]))
            p_next = self.p_dist(sp, ap)
            m = categorical_projection(r, p_next, self.atoms, self.gamma, False)
        p_cur = self.p_dist(s, a)
        if self.loss == "cramer":
            g = cramer_grad_logits(p_cur, m)
        else:
            g = kl_grad_logits(p_cur, m)
        self.logits[s, a] -= self.lr * g
        return p_cur, m

def mc_return_distribution(learner, p_deploy=0.10, n_ep=400, seed=0, T=200):
    rng = np.random.default_rng(seed)
    Rs = []
    for _ in range(n_ep):
        env = ChainMDP(p_slip=p_deploy, rng=np.random.default_rng(rng.integers(1e9)))
        s = env.reset(); G = 0.0
        for _ in range(T):
            qs = [learner.q_value(s, a) for a in range(2)]
            a = int(np.argmax(qs))
            sp, r, done = env.step(a)
            G += r; s = sp
            if done: break
        Rs.append(G)
    return np.array(Rs)

def greedy_eval(learner, p_deploy=0.10, n_ep=60, seed=0):
    rng = np.random.default_rng(seed)
    Rs = []
    for _ in range(n_ep):
        env = ChainMDP(p_slip=p_deploy, rng=np.random.default_rng(rng.integers(1e9)))
        s = env.reset(); G = 0.0
        for _ in range(200):
            qs = [learner.q_value(s, a) for a in range(2)]
            a = int(np.argmax(qs))
            sp, r, done = env.step(a)
            G += r; s = sp
            if done: break
        Rs.append(G)
    return np.array(Rs)

print("learner & eval ready")


learner & eval ready


In [4]:
def train_categorical(loss, K, seed, T_steps=800):
    env = ChainMDP(rng=np.random.default_rng(seed * 11 + 3))
    lr = CatLearner(seed=seed, K=K, loss=loss)
    ent_track = []; proj_track = []
    s = env.reset()
    for t in range(T_steps):
        a = lr.act(s)
        sp, r, done = env.step(a)
        p_cur, m = lr.update(s, a, r, sp, done)
        ent_track.append(entropy(p_cur))
        proj_track.append(float(-np.sum(m * np.log(p_cur + 1e-12))))
        s = sp if not done else env.reset()
    return lr, np.array(ent_track), np.array(proj_track)

Ks = [10, 21, 51]
losses = ["cramer", "kl"]
seeds = [93201, 93202, 93203]

recs = []
tracks = {}
for K in Ks:
    for loss in losses:
        for seed in seeds:
            lr, ent, proj = train_categorical(loss, K, seed)
            H_tail = float(ent[-200:].mean())
            P_tail = float(proj[-200:].mean())
            R = greedy_eval(lr, seed=seed + 5000)
            tail8 = float(R[-8:].mean())
            recs.append({"K": K, "loss": loss, "seed": seed,
                         "H_tail": H_tail, "proj_tail": P_tail, "tail8": tail8})
            tracks[(K, loss, seed)] = (ent, proj)

df = pd.DataFrame(recs)
summary = df.groupby(["K", "loss"], as_index=False).agg(
    H_mean=("H_tail", "mean"), H_std=("H_tail", "std"),
    P_mean=("proj_tail", "mean"), P_std=("proj_tail", "std"),
    R_mean=("tail8", "mean"),   R_std=("tail8", "std"),
)
print(summary)


    K    loss  H_mean  H_std  P_mean  P_std  R_mean  R_std
0  10  cramer  1.5726 0.1142  1.4120 0.0930  0.9250 0.0090
1  10      kl  2.2870 0.0014  1.9835 0.0445  0.8358 0.1611
2  21  cramer  1.7334 0.2460  1.5814 0.2336  0.9250 0.0090
3  21      kl  3.0342 0.0007  2.7399 0.0313  0.6942 0.0416
4  51  cramer  2.2333 0.0954  2.2398 0.1980  0.9250 0.0090
5  51      kl  3.9266 0.0007  3.5786 0.0694  0.1983 0.4589


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

ax = axes[0]
for loss, color in zip(losses, ["#d62728", "#1f77b4"]):
    ents = [tracks[(21, loss, s)][0] for s in seeds]
    m = np.mean(ents, axis=0)
    ax.plot(m, label=f"{loss} (K=21)", color=color, alpha=0.9)
ax.set_xlabel("Training step")
ax.set_ylabel("H(p_hat)")
ax.set_title("Sharpness trajectory (K=21)")
ax.grid(True, alpha=0.3); ax.legend()

ax = axes[1]
Ks_arr = np.array(Ks, dtype=float)
w = 0.35
for i, loss in enumerate(losses):
    sub = summary[summary.loss == loss].sort_values("K")
    ax.bar(Ks_arr + (i - 0.5) * w * 5, sub["R_mean"], w * 5, yerr=sub["R_std"],
           label=loss, capsize=4, edgecolor="black",
           color="#d62728" if loss == "cramer" else "#1f77b4")
ax.set_xticks(Ks_arr)
ax.set_xlabel("Number of atoms K")
ax.set_ylabel("tail-8 return")
ax.set_title("Evaluation return by K x loss")
ax.grid(True, axis="y", alpha=0.3); ax.legend()

fig.tight_layout()
plt.savefig("/tmp/repo/Day93/_p2_bellman.png", dpi=90, bbox_inches="tight")
plt.show()


In [6]:
K_ref = 21
best_row = summary[summary.K == K_ref].sort_values("R_mean", ascending=False).iloc[0]
worst_row = summary[summary.K == K_ref].sort_values("R_mean", ascending=False).iloc[-1]
print(f"K={K_ref} winner: {best_row['loss']} (R={best_row['R_mean']:.3f})")
print(f"K={K_ref} loser:  {worst_row['loss']} (R={worst_row['R_mean']:.3f})")

lr_w, _, _ = train_categorical(best_row["loss"], K_ref, seeds[0])
lr_l, _, _ = train_categorical(worst_row["loss"], K_ref, seeds[0])
mc_returns = mc_return_distribution(lr_w, n_ep=600, seed=999)

fig, ax = plt.subplots(figsize=(8, 4))
atoms = np.linspace(-0.5, 1.0, K_ref)
a_star_w = int(np.argmax([lr_w.q_value(0, a) for a in range(2)]))
a_star_l = int(np.argmax([lr_l.q_value(0, a) for a in range(2)]))
p_w = lr_w.p_dist(0, a_star_w)
p_l = lr_l.p_dist(0, a_star_l)
ax.bar(atoms - 0.015, p_w, width=0.025, label=f"{best_row['loss']} learned p(s0,a*)",
       color="#1f77b4", alpha=0.8, edgecolor="black")
ax.bar(atoms + 0.015, p_l, width=0.025, label=f"{worst_row['loss']} learned p(s0,a*)",
       color="#d62728", alpha=0.8, edgecolor="black")
ax.hist(mc_returns, bins=30, density=True, alpha=0.3, color="gray",
        label="MC empirical return")
ax.set_xlabel("Return atom / value")
ax.set_ylabel("prob / density")
ax.set_title("Learned atom dist vs MC empirical (K=21, s=0)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.savefig("/tmp/repo/Day93/_p2_mc.png", dpi=90, bbox_inches="tight")
plt.show()


K=21 winner: cramer (R=0.925)
K=21 loser:  kl (R=0.694)


## 4. 결과 해석

1. **Sharpness $H(\hat p)$**: Bellman loop 안에서 두 손실의 최종 원자 분포 뾰족함. Day 92
   stationary 결과 (KL 이 더 sharp) 방향이 같은지.
2. **Projection fit (cross-entropy)**: target $m$ 과의 fit 정도. Cramér 는 CDF 를 정합, KL 은
   확률질량을 직접 정합.
3. **Evaluation return**: 실제 의사결정 유용성. 학습 신호가 좋다고 배포 성능이 좋은 건 아님.
4. **K 의존성**: K=10→21→51 로 원자를 늘렸을 때 순위 유지 여부. Cramér 은 원자가 촘촘해질수록
   차이가 완화될 것으로 예상.

> **결론**: Bellman-projection 하에서 두 손실의 순위는 [K 별로 위 표 참조]. Day 92 stationary
> sandbox 의 KL 우위는 [real-net 학습으로 전이 · 부분 전이 · 전이 실패] 관측. K 증가와 함께
> 두 손실의 [수렴 / 발산] 패턴.

### 다음 문제로 연결
P3 는 P1 의 dominant driver 를 uniform multi-slip curriculum 과 결합, Day 91 §15.24 P3
negative finding 의 reversal 여부 판정.
